# Thesis Note — Milestone 7: SSL-Based Multimodal Fusion

**Project:** *Self-Supervised Multimodal Representation Learning for Robust Dental Diagnosis with Naturally Missing Radiographs: A Study on the COde Dataset*
**Milestone:** 7 — Multimodal Fusion
**Status:** Completed as an experimental fusion milestone; missing-modality robustness is intentionally deferred to Milestone 8.
**Date:** 2026-08-29

---

## 1. Purpose of This Milestone

Milestone 7 evaluates whether the modality-specific representations learned during self-supervised multimodal pretraining can be combined into a single downstream classifier for dental multi-label diagnosis.

The pipeline is:

```text
Patient-level dataset + six-label experimental subset
                        |
                        v
              Pretrained SSL Encoders
                /        |        \
               /         |         \
      Photograph     Radiograph    Clinical Text
           \             |             /
            \            |            /
             +------ Representation ------+
                         |
                         v
                  Multimodal Fusion
                         |
                         v
                Multi-label Classifier
                         |
                         v
                     6 Labels
```

The experimental procedure is:

1. Start from the patient-level dataset and the six-label experimental subset.
2. Use the pretrained SSL encoders to extract representations for:

   * Intraoral photographs
   * Radiographs
   * Clinical text
3. Fuse the three representations.
4. Train a multi-label classifier on the fused representation.
5. Compare two fusion architectures:

   * **Main fusion model**
   * **Simple fusion baseline**
6. Evaluate the fusion models under a controlled **complete-case protocol**, where all three modalities are available.

### Central Research Question

> **Can SSL-learned representations from photographs, radiographs, and clinical text be jointly exploited for downstream dental diagnosis?**

This milestone does **not** yet test robustness to naturally missing radiographs. That is the purpose of **Milestone 8**.

---

# 2. Experimental Scope and Protocol

## 2.1 Label Setting

The current fusion experiments use the following six-label experimental subset:

1. Gingivitis
2. Tooth Structure Loss
3. Pulpitis
4. Tooth Loss
5. Dental Caries
6. Malocclusion

The dataset protocol checks that each fusion sample contains:

* At least one photograph
* At least one radiograph
* Non-empty clinical text
* Six binary labels

---

## 2.2 Complete-Case Dataset

For the complete-case experiment, the dataset contains:

| Split      |    Visits |
| ---------- | --------: |
| **Total**  | **4,195** |
| Train      |     2,935 |
| Validation |       627 |
| Test       |       633 |

Only complete cases are used in this fusion experiment.

Therefore, the test set contains **633 samples**, not the full patient-level test split.

This restriction is deliberate: it isolates the question of **multimodal fusion** before introducing missing-modality handling.

---

## 2.3 Patient-Level Split

The project continues to use the previously established **patient-level split with seed 42**.

The fusion stage does not replace or redefine the authoritative patient-level split.

---

## 2.4 Data Leakage Considerations

The six labels originate from the reconstructed-label pipeline.

The fusion downstream task uses the permitted clinical text representation rather than feeding label-generating fields such as `anomalies_en` into the model.

The fusion stage therefore preserves the earlier leakage-avoidance principle:

* `anomalies_en` is **not** used as a predictive text input.
* `diagnosis` is **not** used as a predictive text input.
* Post-diagnosis fields such as `treatment` / `management` are **not** used as predictive text inputs.

---

# 3. SSL Representation Extraction

The multimodal SSL model was trained with a dynamic multimodal contrastive objective using three modality pairs:

* **Image ↔ Text**
* **Image ↔ Radiograph**
* **Radiograph ↔ Text**

The downstream fusion stage consumes the resulting **modality-specific representations** rather than raw images/text.

### Representation Dimensions

| Modality      | Representation Dimension |
| ------------- | -----------------------: |
| Photograph    |                     2048 |
| Radiograph    |                     2048 |
| Clinical Text |                      768 |

Thus, the fusion model receives three learned representations corresponding to the same patient visit.

---

# 4. SSL Alignment Sanity Check

The similarity analysis provides evidence that the SSL pretraining produced meaningful cross-modal alignment.

| Modality Pair    | Positive Mean Similarity | Negative Mean Similarity | Positive Count | Negative Count |
| ---------------- | -----------------------: | -----------------------: | -------------: | -------------: |
| Image–Text       |                   0.4609 |                   0.0791 |          1,313 |          1,330 |
| Image–Radiograph |                   0.4645 |                   0.0926 |            642 |          1,330 |
| Radiograph–Text  |                   0.3887 |                   0.0577 |            627 |          1,330 |

The positive pairs consistently have substantially higher similarity than negative pairs.

The separation is particularly clear for:

* **Image–Text:** `0.4609` vs. `0.0791`
* **Image–Radiograph:** `0.4645` vs. `0.0926`
* **Radiograph–Text:** `0.3887` vs. `0.0577`

This is an important sanity check because the purpose of the SSL stage was not merely to optimize a contrastive loss, but to produce representations in which semantically corresponding modalities are closer than mismatched modalities.

### Interpretation

The results support the claim that the SSL pretraining learned **cross-modal structure**.

However, similarity separation alone is not sufficient to establish downstream diagnostic benefit. That question is addressed by the downstream and fusion experiments below.

---

# 5. Downstream Performance of Individual SSL Representations

Before evaluating fusion, each SSL modality representation was evaluated independently on the same complete-case test protocol.

| SSL Downstream Model |   Macro F1 |   Micro F1 |      AUROC |   Accuracy |
| -------------------- | ---------: | ---------: | ---------: | ---------: |
| **SSL Image**        |     0.5061 |     0.7250 |     0.8372 |     0.6477 |
| **SSL Radiograph**   |     0.2603 |     0.5966 |     0.8043 |     0.5466 |
| **SSL Text**         | **0.7671** | **0.8672** | **0.9522** | **0.7915** |

The **SSL text representation is clearly the strongest individual modality** in this experiment.

This is consistent with the broader baseline findings, where clinical text carries substantial diagnostic information.

Consequently, fusion performance must be interpreted against a **strong text-only reference** rather than assuming that every additional modality will automatically improve all metrics.

---

# 6. Multimodal Fusion Models

Two fusion variants were implemented.

## 6.1 Main Fusion Model

The main architecture combines the three SSL representations and feeds the fused representation into the downstream classifier.

### Training Configuration

| Parameter                | Value  |
| ------------------------ | ------ |
| Seed                     | 42     |
| Batch Size               | 64     |
| Maximum Epochs           | 50     |
| Learning Rate            | 1e-4   |
| Weight Decay             | 1e-4   |
| Device                   | CUDA   |
| Labels                   | 6      |
| Best Epoch               | 41     |
| Best Validation Macro F1 | 0.7514 |

---

## 6.2 Simple Fusion Baseline

A simpler fusion architecture is trained under the same general protocol.

### Configuration

| Parameter                | Value  |
| ------------------------ | ------ |
| Seed                     | 42     |
| Batch Size               | 64     |
| Maximum Epochs           | 50     |
| Learning Rate            | 1e-4   |
| Weight Decay             | 1e-4   |
| Device                   | CUDA   |
| Labels                   | 6      |
| Best Epoch               | 49     |
| Best Validation Macro F1 | 0.7354 |

The simple model is useful as an architectural baseline because it tests whether the multimodal benefit can be obtained without relying on a more elaborate fusion design.

---

# 7. Final Fusion Test Results

## 7.1 Main Fusion Model

| Metric       | Test Result |
| ------------ | ----------: |
| Test Samples |         633 |
| **Macro F1** |  **0.7676** |
| **Micro F1** |  **0.8760** |
| **AUROC**    |  **0.9646** |
| **Accuracy** |  **0.8120** |

---

## 7.2 Simple Fusion Model

| Metric       | Test Result |
| ------------ | ----------: |
| Test Samples |         633 |
| **Macro F1** |  **0.7656** |
| **Micro F1** |  **0.8654** |
| **AUROC**    |  **0.9685** |
| **Accuracy** |  **0.7962** |

---

## 7.3 Direct Comparison

| Model                 |   Macro F1 |   Micro F1 |      AUROC |   Accuracy |
| --------------------- | ---------: | ---------: | ---------: | ---------: |
| SSL Image             |     0.5061 |     0.7250 |     0.8372 |     0.6477 |
| SSL Radiograph        |     0.2603 |     0.5966 |     0.8043 |     0.5466 |
| SSL Text              | **0.7671** | **0.8672** | **0.9522** | **0.7915** |
| SSL Fusion — Simple   |     0.7656 |     0.8654 | **0.9685** |     0.7962 |
| **SSL Fusion — Main** | **0.7676** | **0.8760** |     0.9646 | **0.8120** |

---

# 8. Main Findings

## Finding 1 — SSL Representations Are Cross-Modally Aligned

The similarity analysis shows a clear positive-vs-negative separation for all three modality pairs.

This supports the first downstream assumption of the project: the SSL encoder is producing representations that preserve **cross-modal correspondence**.

---

## Finding 2 — Clinical Text Is the Strongest Individual SSL Modality

Among the three individual downstream models, text obtains:

* **Macro F1 = 0.7671**
* **Micro F1 = 0.8672**
* **AUROC = 0.9522**

Therefore, text is already a very strong predictor in this dataset.

---

## Finding 3 — Fusion Reaches or Slightly Exceeds the Text-Only SSL Baseline

The main fusion model obtains:

| Metric   | Main Fusion | SSL Text | Difference |
| -------- | ----------: | -------: | ---------: |
| Macro F1 |  **0.7676** |   0.7671 |    +0.0005 |
| Micro F1 |  **0.8760** |   0.8672 |    +0.0088 |
| AUROC    |  **0.9646** |   0.9522 |    +0.0124 |
| Accuracy |  **0.8120** |   0.7915 |    +0.0205 |

The simple fusion model shows a similar pattern.

Therefore, multimodal fusion is not merely reproducing a weak unimodal baseline; it produces a strong downstream classifier and improves several aggregate metrics over SSL text-only.

---

## Finding 4 — The Two Fusion Architectures Are Very Close in Macro F1

### Main

**Macro F1 = 0.7676**

### Simple

**Macro F1 = 0.7656**

The difference is only about **0.0020**.

This means the current experiments do not provide strong evidence that the more complex fusion design is substantially better than the simple fusion baseline in Macro F1.

Interestingly, the simple model achieves the highest AUROC:

* **Simple = 0.9685**
* **Main = 0.9646**

Therefore, the main architecture should **not** be described as universally superior.

Its clearest advantage in this experiment is **Micro F1 and accuracy**, while the simple model has slightly higher **AUROC**.

---

# 9. Relationship to the Previous Baseline Experiments

The earlier baseline experiments showed that clinical text is highly informative and that the full multimodal baseline performs strongly.

The Milestone 7 experiment asks a different question:

> **What happens when the downstream classifier uses representations learned by multimodal SSL rather than relying directly on independently trained modality encoders?**

The current result shows that SSL-based fusion reaches strong performance despite using learned representations rather than training the complete multimodal network from scratch at the fusion stage.

The comparison should nevertheless be made carefully because the Milestone 7 experiment uses the **six-label complete-case protocol with 633 test samples**, while the earlier full baseline table used a different experimental scope.

They should **not** be presented as if they were identical benchmark conditions.

---

# 10. What This Milestone Proves

Milestone 7 provides evidence for the following:

1. The multimodal SSL model produces aligned representations across the three modalities.
2. Each modality representation can support downstream multi-label diagnosis.
3. Combining the SSL representations produces a strong multimodal classifier.
4. The fused model performs at least comparably to the strong SSL text-only model and improves **Micro F1, AUROC, and accuracy** in the main comparison.
5. A relatively simple fusion strategy is already competitive with the main fusion architecture.

---

# 11. What This Milestone Does **NOT** Prove

> **This distinction is critical for the thesis.**

Milestone 7 does **not yet prove robustness to missing radiographs**.

The current fusion evaluation requires:

* Photograph available
* Radiograph available
* Clinical text available

Therefore, the **633-test-sample fusion result is a complete-case multimodal result**.

It cannot by itself support the thesis claim that the model is robust when radiographs are naturally absent.

That claim must be tested separately.

---

# 12. Why Milestone 8 Is Necessary

The COde dataset contains substantial natural variation in radiograph availability.

The earlier dataset audit showed that radiographs are available for only about half of visits, while photographs and clinical text are almost universally available.

This creates the central experimental opportunity of the thesis:

> **Can a model trained using multimodal information maintain useful diagnostic performance when the radiograph modality is unavailable?**

Milestone 8 should therefore compare at least:

1. **Complete multimodal input**
2. **Natural missing-radiograph input**
3. **Controlled missing-radiograph input**, where appropriate for a matched comparison

The key requirement is that missing-radiograph evaluation must be performed with a model and inference protocol that explicitly support missing modalities.

The current complete-case fusion result should be treated as the **reference point**, not as the robustness experiment itself.

---

# 13. Limitations of the Current Milestone

## 13.1 Complete-Case Restriction

The fusion experiment evaluates only **633 test visits** because all three modalities are required.

This is appropriate for isolating fusion quality but does not reflect the full naturally occurring test distribution.

---

## 13.2 Strong Text Modality

Clinical text is highly predictive in this dataset.

Consequently, multimodal fusion gains are naturally constrained: the model starts from a very strong text-only representation.

---

## 13.3 No Missing-Modality Mechanism Yet

The current fusion model expects all three modality representations.

It therefore cannot yet be interpreted as a **missing-modality-robust model**.

---

## 13.4 Architecture Comparison Is Not Decisive

Main and Simple fusion are extremely close in Macro F1.

The current results support keeping both as the principal fusion architecture and simple baseline, but they do not justify a strong claim that the main architecture is substantially superior.

---

## 13.5 Metric Interpretation

Because this is a **multi-label classification task**, Macro F1 should remain a central metric.

Micro F1 and AUROC are useful complementary metrics but should not be used alone to characterize performance.

---

# 14. Milestone 7 Deliverables

## Completed

* [x] Defined the six-label fusion downstream task.
* [x] Validated the complete-case fusion dataset protocol.
* [x] Extracted SSL representations for photographs, radiographs, and clinical text.
* [x] Verified positive-vs-negative cross-modal similarity separation.
* [x] Implemented downstream evaluation for individual SSL modalities.
* [x] Implemented the main multimodal fusion model.
* [x] Implemented the simple fusion baseline.
* [x] Trained both fusion variants.
* [x] Evaluated both on the complete-case test set.
* [x] Saved configuration, training history, and test metrics.
* [x] Established the complete-case fusion result as the reference for the missing-modality experiment.

---

# 15. Artifacts Produced

## Fusion

```text
results/fusion/main/config.json
results/fusion/main/history.json
results/fusion/main/test_metrics.json

results/fusion/simple/config.json
results/fusion/simple/history.json
results/fusion/simple/test_metrics.json
```

## SSL Alignment

```text
results/ssl_pretraining/similarity_analysis/similarity_results.json
```

## SSL Downstream

```text
results/ssl_pretraining/downstream/image/history.json
results/ssl_pretraining/downstream/radiograph/history.json
results/ssl_pretraining/downstream/text/history.json
```

## Complete-Case Reference Experiment

```text
results/ssl_pretraining/test_complete_case_4195/complete_case_dataset.csv

results/ssl_pretraining/test_complete_case_4195/image/fine_tune/test_metrics.json
results/ssl_pretraining/test_complete_case_4195/radiograph/fine_tune/test_metrics.json
results/ssl_pretraining/test_complete_case_4195/text/fine_tune/test_metrics.json
```

---

# 16. Thesis-Ready Result Statement

> The multimodal SSL representations were evaluated through a downstream fusion task using six reconstructed dental diagnostic labels. Cross-modal similarity analysis demonstrated clear separation between matched and mismatched modality pairs, indicating that the self-supervised pretraining captured meaningful multimodal correspondence. On the complete-case test set of 633 visits, the main fusion model achieved a Macro F1 of 0.7676, Micro F1 of 0.8760, AUROC of 0.9646, and accuracy of 0.8120. A simple fusion baseline produced comparable results, with Macro F1 of 0.7656, Micro F1 of 0.8654, AUROC of 0.9685, and accuracy of 0.7962. The strong SSL text-only model achieved Macro F1 of 0.7671, Micro F1 of 0.8672, and AUROC of 0.9522. These results indicate that SSL-based multimodal fusion provides a strong downstream representation and improves several aggregate metrics relative to the text-only SSL representation. However, because the fusion evaluation was restricted to complete cases, these experiments do not yet establish robustness to naturally missing radiographs; this question is addressed separately in the missing-modality experiments.

---

# 17. Final Milestone Conclusion

**Milestone 7 is considered experimentally complete.**

The evidence is sufficient to move forward:

```text
SSL Pretraining
       |
       v
Aligned Modality Representations
       |
       v
Unimodal Downstream Validation
       |
       v
Complete-Case Multimodal Fusion
```

The next milestone should **not redesign the SSL or fusion pipeline** unless a concrete implementation problem is discovered.

The immediate research question is now:

> **How does the trained multimodal system behave when radiographs are naturally missing, and can the proposed representation/fusion strategy preserve diagnostic performance under that condition?**

That question constitutes:

# Milestone 8 — Missing-Modality Robustness

---


# Thesis Note — Diagnostic End-to-End SSL Fusion Fine-Tuning

## 1. Experiment Overview

This experiment was conducted as a diagnostic extension of the main multimodal fusion experiment to determine whether the performance of the SSL-based fusion model was limited by the use of frozen/precomputed SSL representations.

In the main SSL Fusion pipeline, the representations learned during self-supervised pretraining were extracted and used as fixed inputs to the downstream fusion network. In this diagnostic experiment, the SSL encoders were instead kept inside the computational graph and jointly fine-tuned with the existing MainFusion architecture.

The purpose was therefore **not to replace the main Fusion experiment**, but to test whether unrestricted end-to-end optimization of the pretrained encoders could improve downstream diagnostic performance.

The experimental pipeline was:

```text
Photographs ───────→ SSL Image Encoder ────────┐
                                               │
Radiographs ───────→ SSL Radiograph Encoder ───┤
                                               │
Clinical Text ─────→ SSL Text Encoder ─────────┤
                                               ↓
                                          MainFusion
                                               ↓
                                           6 Labels
```

Unlike the original Fusion experiment, the three SSL encoders were trainable during downstream classification.

The SSL projection heads were not used.

---

## 2. Experimental Protocol

The experiment used the same complete-case population and the same authoritative patient-level split as the existing Fusion experiments.

### Dataset

* Source dataset: six-label patient-level labeled dataset
* Task: six-label multi-label classification
* Complete-case population: **4,195 visits**
* Training set: **2,935 visits**
* Validation set: **627 visits**
* Test set: **633 visits**

The complete-case population was explicitly verified before training.

```text
Source rows:             8,775
Complete-case visits:    4,195

Train:                   2,935
Validation:                627
Test:                      633
Total:                   4,195
```

The split was:

```text
Train      2,935
Validation   627
Test         633
```

No changes were made to the authoritative patient-level split.

### Model

The experiment initialized the multimodal encoders from:

`best_ssl_model.pt`

The architecture consisted of:

* SSL Image Encoder → 2048-dimensional representation
* SSL Radiograph Encoder → 2048-dimensional representation
* SSL Text Encoder → 768-dimensional representation
* MainFusion downstream classifier
* Six output labels

The SSL projection heads were excluded from downstream classification.

### Fine-Tuning Configuration

* Batch size: **8**
* Gradient accumulation steps: **8**
* Effective batch size: **64**
* Maximum epochs: **70**
* Learning rate: **1 × 10⁻⁵**
* Weight decay: **1 × 10⁻⁴**
* Early stopping based on validation Macro F1
* Initial SSL weights: best multimodal SSL checkpoint

The model contained approximately **119.35 million parameters**, of which approximately **116.66 million parameters were trainable**.

---

## 3. Training Behavior

The experiment demonstrated a strong tendency toward overfitting.

During training, the training loss decreased to extremely small values:

```text
Epoch 58:
Train loss ≈ 0.0016
```

However, validation performance did not improve consistently with the decreasing training loss.

The best validation performance was obtained at Epoch 58:

```text
Validation Macro F1 = 0.6886
Validation Micro F1  = 0.8531
Validation AUROC     = 0.9384
Validation Accuracy  = 0.7959
```

After this point, the training loss remained very low while validation performance fluctuated and generally deteriorated.

For example:

```text
Epoch 58
Train loss           0.0016
Validation loss      0.2308
Validation Macro F1  0.6886
Validation AUROC     0.9384

Epoch 70
Train loss           0.0029
Validation loss      0.2574
Validation Macro F1  0.6720
Validation AUROC     0.9336
```

This behavior indicates that the model was increasingly fitting the training data without obtaining corresponding improvements in validation generalization.

The observed behavior is consistent with overfitting caused by unrestricted fine-tuning of a very large pretrained multimodal model on a relatively small complete-case downstream dataset.

---

## 4. Final Test Results

After early stopping, the best validation checkpoint was restored and evaluated on the held-out test set.

The best checkpoint corresponded to:

**Epoch 58**

Final test performance:

| Metric   |       Test |
| -------- | ---------: |
| Macro F1 | **0.7342** |
| Micro F1 | **0.8482** |
| AUROC    | **0.9331** |
| Accuracy | **0.7741** |

The complete test population consisted of **633 visits**.

---

## 5. Comparison with Frozen SSL Fusion

The key purpose of this experiment was to determine whether fine-tuning the SSL encoders could improve the existing frozen-representation Fusion model.

The previous SSL Fusion results were:

| Model                             |   Macro F1 |   Micro F1 |      AUROC |   Accuracy |
| --------------------------------- | ---------: | ---------: | ---------: | ---------: |
| SSL Image                         |     0.5061 |     0.7250 |     0.8372 |     0.6477 |
| SSL Radiograph                    |     0.2603 |     0.5966 |     0.8043 |     0.5466 |
| SSL Text                          | **0.7671** | **0.8672** | **0.9522** | **0.7915** |
| SSL Fusion — Simple               |     0.7656 |     0.8654 | **0.9685** |     0.7962 |
| **SSL Fusion — Main (Frozen)**    | **0.7676** | **0.8760** |     0.9646 | **0.8120** |
| SSL Fusion — Main (End-to-End FT) |     0.7342 |     0.8482 |     0.9331 |     0.7741 |

The direct comparison between the Main Fusion configurations is:

| Metric   | Frozen Main | End-to-End FT |  Difference |
| -------- | ----------: | ------------: | ----------: |
| Macro F1 |  **0.7676** |        0.7342 | **−0.0334** |
| Micro F1 |  **0.8760** |        0.8482 | **−0.0278** |
| AUROC    |  **0.9646** |        0.9331 | **−0.0315** |
| Accuracy |  **0.8120** |        0.7741 | **−0.0379** |

Thus, end-to-end fine-tuning did not improve any of the evaluated test metrics.

---

## 6. Interpretation

The diagnostic experiment provides evidence that the performance of the main SSL Fusion model is **not primarily limited by freezing the pretrained SSL representations**.

If frozen representations had been the main bottleneck, allowing all SSL encoders to adapt jointly to the downstream classification task would be expected to produce a meaningful improvement.

Instead, the opposite behavior was observed.

The frozen SSL Fusion model achieved:

* Macro F1 = **0.7676**
* Micro F1 = **0.8760**
* AUROC = **0.9646**
* Accuracy = **0.8120**

while end-to-end fine-tuning achieved:

* Macro F1 = **0.7342**
* Micro F1 = **0.8482**
* AUROC = **0.9331**
* Accuracy = **0.7741**

The reduction across all four metrics, together with the very low training loss and deteriorating validation behavior, suggests that unrestricted fine-tuning caused over-specialization to the downstream training data.

A likely contributing factor is the large number of trainable parameters relative to the size of the complete-case training population. The experiment optimized approximately **116.66 million trainable parameters using only 2,935 training visits**.

Therefore, the pretrained SSL representations appear to provide sufficiently informative and transferable features for the downstream Fusion task, while unrestricted joint fine-tuning is not beneficial under the current data regime.

---

## 7. Decision for the Main Thesis Pipeline

Based on this diagnostic experiment, **end-to-end fine-tuning is not selected as the main Fusion configuration**.

The primary Fusion model remains:

**SSL Fusion — Main (Frozen)**

with the following test performance:

```text
Macro F1 : 0.7676
Micro F1 : 0.8760
AUROC    : 0.9646
Accuracy : 0.8120
```

The end-to-end fine-tuning experiment is retained as a **diagnostic/ablation experiment** demonstrating that unrestricted downstream adaptation of the SSL encoders does not improve performance and instead leads to poorer generalization.

This result supports the use of the pretrained SSL representations as fixed multimodal representations for the subsequent robustness analysis.

---

## 8. Relevance to the Thesis Research Question

The result is particularly relevant because the central objective of the thesis is not simply to maximize complete-case classification performance, but to investigate whether self-supervised multimodal representations can support robust dental diagnosis when one or more modalities are unavailable.

The diagnostic fine-tuning experiment therefore establishes the downstream configuration to be used in the next stage:

```text
Multimodal SSL Pretraining
          ↓
Frozen SSL Encoders
          ↓
Main Fusion
          ↓
Complete-Case Evaluation
          ↓
Missing-Modality Robustness Evaluation
```

The next experimental stage should therefore focus on **Missing-Modality Robustness**, rather than further unrestricted fine-tuning of the SSL encoders.

---

## 9. Reproducibility Information

**Experiment name**

`end_to_end_ssl_fusion_finetune`

**Dataset**

`results/six_label_patient_level_dataset/labeled_dataset.csv`

**SSL checkpoint**

`results/ssl_pretraining/multimodal_dynamic/best_ssl_model.pt`

**Output directory**

`results/fusion/main_finetune/`

**Complete-case population**

4,195 visits

**Split**

2,935 / 627 / 633

**Best epoch**

58

**Best validation Macro F1**

0.6886

**Best validation AUROC**

0.9384

**Test Macro F1**

0.7342

**Test Micro F1**

0.8482

**Test AUROC**

0.9331

**Test Accuracy**

0.7741

**Trainable parameters**

116,660,870

**Projection heads**

Not used

**Missing-modality experiments**

Not included

**Role in thesis**

Diagnostic / ablation experiment; not the primary Fusion model.
